<a href="https://colab.research.google.com/github/danieltalon/colab-python/blob/main/PID_Desafio02_Exp05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ipywidgets import interact, widgets

# 1. Carregar as Imagens
def carregar_imagem(caminho):
    img = Image.open(caminho).convert('L')
    return np.array(img, dtype=np.float64)

img_ideal = carregar_imagem('imagem_b.png')
img_real = carregar_imagem('capturada_b_v4.jpg')

# 2. Funções de Cálculo
def extrair_perfil(imagem, coord, orientacao):
    return imagem[coord, :] if orientacao == 'horizontal' else imagem[:, coord]

def suavizar_perfil(perfil, tamanho_janela=51):
    suavizado = np.zeros_like(perfil)
    offset = tamanho_janela // 2
    for i in range(len(perfil)):
        inicio, fim = max(0, i - offset), min(len(perfil), i + offset + 1)
        suavizado[i] = np.sum(perfil[inicio:fim]) / (fim - inicio)
    return suavizado

# 3. Função Principal de Visualização
def analisar_tudo(coord, orientacao):
    # Extração e Processamento
    p_ideal = extrair_perfil(img_ideal, coord, orientacao)
    p_real = extrair_perfil(img_real, coord, orientacao)
    p_real_suav = suavizar_perfil(p_real, tamanho_janela=51)

    # Cálculos de LSF
    lsf_bruta = np.diff(p_real)
    lsf_suav = np.abs(np.diff(p_real_suav))

    # Limites ESF
    val_min, val_max = np.min(p_real_suav), np.max(p_real_suav)
    lim_10, lim_90 = val_min + 0.1*(val_max-val_min), val_min + 0.9*(val_max-val_min)

    # Encontrar índices
    #idx_10 = np.where(p_real_suav >= lim_10)[0][0]
    #idx_90 = np.where(p_real_suav >= lim_90)[0][0]
    #delta_x = abs(idx_90 - idx_10)

    # Encontrar índices de forma robusta
    # Cruzamentos são os índices onde o sinal ultrapassa o nível limiar
    #cruzamentos_10 = np.where(p_real_suav >= lim_10)[0]
    #cruzamentos_90 = np.where(p_real_suav >= lim_90)[0]

    # A maior diferença absoluta na curva suavizada aponta onde a borda está
    gradiente = np.abs(np.diff(p_real_suav))
    centro_borda = np.argmax(gradiente)

    # 2. Definir uma janela de busca (ex: 100 pixels em torno do centro)
    # Isso evita que o ruído no fim da imagem atrapalhe o cálculo
    janela = 30
    inicio_busca = max(0, centro_borda - janela)
    fim_busca = min(len(p_real_suav), centro_borda + janela)

    # 3. Buscar os 10% e 90% APENAS dentro dessa janela
    # Filtramos os cruzamentos que estão dentro da nossa zona de interesse
    cruzamentos_10 = np.where(p_real_suav[inicio_busca:fim_busca] >= lim_10)[0] + inicio_busca
    cruzamentos_90 = np.where(p_real_suav[inicio_busca:fim_busca] >= lim_90)[0] + inicio_busca

    # Lógica condicional: Se o primeiro valor for maior que o último, é uma descida
    if p_real_suav[0] > p_real_suav[-1]:
        # Para descida: 90% acontece primeiro (índice menor), 10% acontece depois (índice maior)
        idx_90 = cruzamentos_90[0]
        idx_10 = cruzamentos_10[-1]
    else:
        # Para subida: 10% acontece primeiro, 90% acontece depois
        idx_10 = cruzamentos_10[0]
        idx_90 = cruzamentos_90[-1]
    delta_x = abs(idx_90 - idx_10)

    # Layout de Gráficos (Grid 2x4)
    fig = plt.figure(figsize=(20, 10))

    # Imagens
    ax0 = plt.subplot(2, 4, 1)
    ax0.imshow(img_ideal, cmap='gray'); ax0.set_title("Imagem Ideal")

    ax1 = plt.subplot(2, 4, 2)
    ax1.imshow(img_real, cmap='gray'); ax1.set_title("Imagem Capturada")
    if orientacao == 'horizontal':
        ax0.axhline(coord, color='red'); ax1.axhline(coord, color='red')
    else:
        ax0.axvline(coord, color='red'); ax1.axvline(coord, color='red')

    # Comparação Direta
    ax2 = plt.subplot(2, 4, 3)
    ax2.plot(p_ideal, label='Ideal', color='blue'); ax2.plot(p_real, label='Capturada', color='orange', alpha=0.5)
    ax2.set_title("Comparação Direta"); ax2.legend(); ax2.grid(True)

    # ESF (Suavizado)
    ax3 = plt.subplot(2, 4, 4)
    ax3.plot(p_real, label='Original', color='orange', alpha=0.3)
    ax3.plot(p_real_suav, label='Suavizado', color='black', linewidth=2)
    ax3.axhline(lim_10, color='green', linestyle=':'); ax3.axhline(lim_90, color='red', linestyle=':')
    ax3.scatter([idx_10, idx_90], [lim_10, lim_90], color=['green', 'red'], zorder=5)
    ax3.set_title(f"ESF: Δx = {delta_x} px"); ax3.grid(True)

    # LSF Bruta
    ax4 = plt.subplot(2, 2, 3)
    ax4.plot(lsf_bruta, color='red', alpha=0.5, label='LSF (Ruído)')
    ax4.set_title("LSF - Sinal Bruto (Com Ruído)"); ax4.grid(True); ax4.legend()

    # LSF Suavizada (h(x))
    ax5 = plt.subplot(2, 2, 4)
    ax5.plot(lsf_suav, color='purple', linewidth=2, label='h(x)')
    ax5.fill_between(range(len(lsf_suav)), lsf_suav, color='purple', alpha=0.2)
    ax5.set_title("LSF - Função de Degradação h(x) Suavizada"); ax5.grid(True); ax5.legend()

    plt.tight_layout()
    plt.show()

# 4. Execução
interact(analisar_tudo,
         coord=widgets.IntSlider(min=0, max=511, step=1, value=256),
         orientacao=widgets.Dropdown(options=['horizontal', 'vertical'], value='vertical'));

interactive(children=(IntSlider(value=256, description='coord', max=511), Dropdown(description='orientacao', i…